In [ ]:
# Imports and display defaults
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (8,4)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

In [ ]:
# Load CSV outputs (same as frontend)
data_dir = Path('outputs')
risk = pd.read_csv(data_dir/'risk_scores.csv')
daily = pd.read_csv(data_dir/'daily_summary.csv', parse_dates=['date'])
state = pd.read_csv(data_dir/'state_summary.csv')
district = pd.read_csv(data_dir/'district_summary.csv')
alerts = pd.read_csv(data_dir/'alerts.csv')

# Basic sanity: non-empty and expected columns
assert not risk.empty, 'risk_scores.csv is empty'
assert {'total_enrollments','total_demo_updates','total_bio_updates'}.issubset(risk.columns)

## Level 0 — Data Familiarization
**Question:** Do we have enough coverage and time-window alignment to trust downstream signals?
**Formula:** completeness = non-null rows ÷ total rows (per column).
**Assumption:** Missingness is random; large gaps mean upstream issues.

In [ ]:
# Completeness per selected columns
cols = ['age_0_5','age_5_17','age_18_greater','total_enrollments','total_demo_updates','total_bio_updates','child_ratio','update_ratio','bio_recapture_ratio']
comp = risk[cols].notna().mean().sort_values()*100
comp.plot(kind='barh', color='#14B8A6')
plt.xlabel('Completeness (%)')
plt.title('Column completeness')
plt.tight_layout()
plt.show()

In [ ]:
# Temporal coverage from daily_summary
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(daily['date'], daily['total_enrollments'], label='Enrollments', color='#14B8A6')
ax.plot(daily['date'], daily['total_demo_updates'], label='Demo updates', color='#E0B15C')
ax.plot(daily['date'], daily['total_bio_updates'], label='Bio updates', color='#6C7FF2')
ax.set_title('Temporal coverage')
ax.legend()
plt.tight_layout()
plt.show()

## Level 1 — Variable-Level Visualization
**Question:** What do raw distributions look like across PINs?
**Formula:** total = age_0_5 + age_5_17 + age_18_greater.
**Assumption:** Age buckets are mutually exclusive and complete.

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(14,4), sharey=False)
metrics = [
    ('total_enrollments', '#14B8A6', 'Enrollments'),
    ('total_demo_updates', '#E0B15C', 'Demo updates'),
    ('total_bio_updates', '#6C7FF2', 'Bio updates'),
]
for ax, (col, color, label) in zip(axes, metrics):
    risk[col].plot(kind='hist', bins=30, color=color, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## Level 2 — Derived Feature Visuals
**Question:** Are updates outpacing enrollments for certain PINs?
**Formula:** update_ratio = total_demo_updates ÷ total_enrollments.
**Assumption:** Same time window; no double-counting.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(risk['total_enrollments'], risk['total_demo_updates'], alpha=0.4, color='#E0B15C', s=12)
ax.set_xlabel('Enrollments')
ax.set_ylabel('Demo updates')
ax.set_title('Update ratio vs enrollments')
plt.tight_layout()
plt.show()

## Level 3 — Geographic Intelligence
**Question:** Which districts/states carry highest average risk?
**Formula:** rank by avg_risk_score; show max_risk_score and volumes.
**Assumption:** Aggregations use consistent codes and periods.

In [ ]:
top_districts = district.sort_values('avg_risk_score', ascending=False).head(5)
display(top_districts[['state','district','avg_risk_score','max_risk_score','total_enrollments','total_demo_updates','total_bio_updates']])

## Level 4 — Temporal & Correlation
**Question:** When did activity spike beyond baseline? What variables co-move?
**Formula:** spike if day-over-day enrollment growth > 2× std; Pearson r for correlation.
**Assumption:** Short-term variance proxies normal baseline; correlations capture linear effects.

In [ ]:
# Spike markers
enroll = daily['total_enrollments']
growth = enroll.pct_change()
threshold = growth.std()*2
spikes = daily[growth > threshold]

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(daily['date'], enroll, color='#14B8A6', label='Enrollments')
ax.scatter(spikes['date'], spikes['total_enrollments'], color='#C44536', label='Spike', zorder=3)
ax.legend()
ax.set_title('Change flags')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
metrics = ['total_enrollments','total_demo_updates','total_bio_updates','update_ratio','bio_recapture_ratio']
corr = risk[metrics].corr(method='pearson')
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(corr, cmap='Blues', vmin=-1, vmax=1)
ax.set_xticks(range(len(metrics)), metrics, rotation=45, ha='right')
ax.set_yticks(range(len(metrics)), metrics)
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Pearson correlation')
plt.tight_layout()
plt.show()

## Level 5 — Anomaly Detection
**Question:** How many PINs have extreme child_ratio? When were high-risk alerts raised?
**Formula:** z = (value - mean) ÷ std; flag if |z| > 3. Y = alert risk_score; color = risk_level.
**Assumption:** Child_ratio roughly normal; alerts deduped per PIN-date.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
(risk['child_ratio_zscore'].dropna()).plot(kind='hist', bins=30, color='#C44536', ax=ax)
ax.axvline(3, color='#E0B15C', linestyle='--')
ax.axvline(-3, color='#E0B15C', linestyle='--')
ax.set_title('Child ratio z-score distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Alerts scatter by date vs risk_score
if not alerts.empty and 'date_detected' in alerts.columns:
    alerts['date_detected'] = pd.to_datetime(alerts['date_detected'], errors='coerce')
    fig, ax = plt.subplots(figsize=(8,4))
    colors = {'CRITICAL':'#C44536','HIGH':'#C44536','MEDIUM':'#E0B15C','LOW':'#6C7FF2'}
    ax.scatter(alerts['date_detected'], alerts['risk_score'], c=alerts['risk_level'].map(colors).fillna('#6C7FF2'), s=16, alpha=0.6)
    ax.axhline(8, color='#C44536', linestyle='--', label='Critical threshold')
    ax.set_ylabel('Risk score')
    ax.set_title('Alerts over time')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No alerts data to plot')

## Level 6 — Risk Scoring & National Synthesis
**Question:** What is the spread of composite risk scores and tier composition by state?
**Formula:** risk_score = 0.30*enrollment_velocity + 0.25*update_velocity + 0.20*demo_z + 0.15*geo_outlier + 0.10*temporal_spike.
**Assumption:** Weights fixed; inputs standardized.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
(risk['risk_score'].dropna()).plot(kind='hist', bins=30, color='#6C7FF2', ax=ax)
ax.set_title('Risk score distribution')
plt.tight_layout()
plt.show()

In [ ]:
# State-level composition (share of high risk vs total pins)
state_plot = state.sort_values('high_risk_pins', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(state_plot['state'], state_plot['high_risk_pins'], color='#C44536', label='High risk pins')
ax.bar(state_plot['state'], state_plot['pin_count'], color='#6C7FF2', alpha=0.35, label='Total pins')
ax.set_xticklabels(state_plot['state'], rotation=45, ha='right')
ax.set_title('Risk tier composition by state')
ax.legend()
plt.tight_layout()
plt.show()